# Lab | Hypothesis Testing

**Objective**

Welcome to the Hypothesis Testing Lab, where we embark on an enlightening journey through the realm of statistical decision-making! In this laboratory, we delve into various scenarios, applying the powerful tools of hypothesis testing to scrutinize and interpret data.

From testing the mean of a single sample (One Sample T-Test), to investigating differences between independent groups (Two Sample T-Test), and exploring relationships within dependent samples (Paired Sample T-Test), our exploration knows no bounds. Furthermore, we'll venture into the realm of Analysis of Variance (ANOVA), unraveling the complexities of comparing means across multiple groups.

So, grab your statistical tools, prepare your hypotheses, and let's embark on this fascinating journey of exploration and discovery in the world of hypothesis testing!

**Challenge 1**

In this challenge, we will be working with pokemon data. The data can be found here:

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv

In [1]:
#libraries
import pandas as pd
import scipy.stats as st
import numpy as np



In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv")
df

,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,Charmander,Fire,NaN,39,52,43,60,50,65,1,False
...,...,...,...,...,...,...,...,...,...,...,...
795,Diancie,Rock,Fairy,50,100,150,100,150,50,6,True
796,Mega Diancie,Rock,Fairy,50,160,110,160,110,110,6,True
797,Hoopa Confined,Psychic,Ghost,80,110,60,150,130,70,6,True
798,Hoopa Unbound,Psychic,Dark,80,160,60,170,130,80,6,True


- We posit that Pokemons of type Dragon have, on average, more HP stats than Grass. Choose the propper test and, with 5% significance, comment your findings.

In [3]:
# Dragon vs Grass (Type 1) - HP
dragon = df[df["Type 1"] == "Dragon"]["HP"]
grass  = df[df["Type 1"] == "Grass"]["HP"]

print("n:", len(dragon), len(grass))
print("mean:", round(dragon.mean(), 2), round(grass.mean(), 2))

# Assumptions
print("Shapiro Dragon:", st.shapiro(dragon))
print("Shapiro Grass :", st.shapiro(grass))
print("Levene        :", st.levene(dragon, grass))

# One-sided two-sample t-test (H1: mean Dragon > mean Grass)
t_stat, p_value = st.ttest_ind(dragon, grass, equal_var=True, alternative="greater")
print("t =", round(t_stat, 3), "| p-value =", p_value)

# Robustness checks
print("Welch       :", st.ttest_ind(dragon, grass, equal_var=False, alternative="greater"))
print("Mann-Whitney:", st.mannwhitneyu(dragon, grass, alternative="greater"))

n: 32 70
mean: 83.31 67.27
Shapiro Dragon: ShapiroResult(statistic=np.float64(0.9635281236696637), pvalue=np.float64(0.3419890019248819))
Shapiro Grass : ShapiroResult(statistic=np.float64(0.9715915656401263), pvalue=np.float64(0.11214356726520774))
Levene        : LeveneResult(statistic=np.float64(1.867317074967163), pvalue=np.float64(0.1748473993433114))
t = 3.59 | p-value = 0.0002567969150153484
Welch       : TtestResult(statistic=np.float64(3.3349632905124063), pvalue=np.float64(0.0007993609745420602), df=np.float64(50.83784116232685))
Mann-Whitney: MannwhitneyuResult(statistic=np.float64(1586.5), pvalue=np.float64(0.0003803459868039855))


**Chosen test:** one-sided two-sample independent t-test.

- H0: mean HP (Dragon) ≤ mean HP (Grass)
- H1: mean HP (Dragon) > mean HP (Grass)

**Conclusion:** p-value ≈ 0.0003 < 0.05, so we reject H0. Dragon Pokémon have significantly higher HP on average than Grass Pokémon (83.3 vs 67.3). The assumptions hold (Shapiro: p = 0.34 and 0.11; Levene: p = 0.17), and the result is confirmed by Welch (p = 0.0008) and Mann-Whitney (p = 0.0004).

- We posit that Legendary Pokemons have different stats (HP, Attack, Defense, Sp.Atk, Sp.Def, Speed) when comparing with Non-Legendary. Choose the propper test and, with 5% significance, comment your findings.


In [4]:
# Legendary vs Non-Legendary - 6 stats
legendary     = df[df["Legendary"] == True]
non_legendary = df[df["Legendary"] == False]

stats_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
results = []

for col in stats_cols:
    levene_p = st.levene(legendary[col], non_legendary[col]).pvalue
    # Welch t-test (robust to unequal variances, e.g. Speed)
    t, p = st.ttest_ind(legendary[col], non_legendary[col], equal_var=False)
    results.append({
        "stat": col,
        "mean_legendary": round(legendary[col].mean(), 1),
        "mean_non_legendary": round(non_legendary[col].mean(), 1),
        "levene_p": levene_p,
        "t_stat": t,
        "p_value": p,
        "reject_H0": p < 0.05,
    })

pd.DataFrame(results)

,stat,mean_legendary,mean_non_legendary,levene_p,t_stat,p_value,reject_H0
0,HP,92.7,67.2,0.575843,8.981370,1.002691e-13,True
1,Attack,116.7,75.7,0.994465,10.438134,2.520372e-16,True
2,Defense,99.7,71.6,0.320343,7.637078,4.826998e-11,True
3,Sp. Atk,122.2,68.5,0.370141,13.417450,1.551461e-21,True
4,Sp. Def,105.9,68.9,0.776411,10.015697,2.294933e-15,True
5,Speed,100.2,65.5,0.001786,11.475044,1.049016e-18,True


**Chosen test:** two-sided independent t-test (Welch version), applied to each stat.

- H0: mean (Legendary) = mean (Non-Legendary)
- H1: mean (Legendary) ≠ mean (Non-Legendary)

**Conclusion:** for all 6 stats, p-value < 0.05, so we reject H0 every time. Legendary Pokémon have significantly different stats, and higher ones, than non-Legendary Pokémon. The gap is largest for Sp. Atk (122.2 vs 68.5).

**Challenge 2**

In this challenge, we will be working with california-housing data. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv

In [5]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


**We posit that houses close to either a school or a hospital are more expensive.**

- School coordinates (-118, 34)
- Hospital coordinates (-122, 37)

We consider a house (neighborhood) to be close to a school or hospital if the distance is lower than 0.50.

Hint:
- Write a function to calculate euclidean distance from each house (neighborhood) to the school and to the hospital.
- Divide your dataset into houses close and far from either a hospital or school.
- Choose the propper test and, with 5% significance, comment your findings.
 

In [6]:
# Euclidean distance to school and hospital
def euclidean_distance(df, lon, lat):
    return np.sqrt((df["longitude"] - lon) ** 2 + (df["latitude"] - lat) ** 2)

df["dist_school"]   = euclidean_distance(df, -118, 34)
df["dist_hospital"] = euclidean_distance(df, -122, 37)

# Close = within 0.5 of a school OR a hospital
is_close = (df["dist_school"] < 0.5) | (df["dist_hospital"] < 0.5)

close = df[is_close]["median_house_value"]
far   = df[~is_close]["median_house_value"]

print("n:", len(close), len(far))
print("mean:", round(close.mean()), round(far.mean()))

n: 6829 10171
mean: 246952 180678


In [7]:
# Assumptions + test
print("Levene:", st.levene(close, far))   # p < 0.05 -> unequal variances -> Welch

# One-sided Welch t-test (H1: mean close > mean far)
t_stat, p_value = st.ttest_ind(close, far, equal_var=False, alternative="greater")
print("t =", round(t_stat, 2), "| p-value =", p_value)

# Robustness check
print("Mann-Whitney:", st.mannwhitneyu(close, far, alternative="greater"))

Levene: LeveneResult(statistic=np.float64(5.34456402848745), pvalue=np.float64(0.020798874811634085))
t = 37.99 | p-value = 1.5032478884295899e-301
Mann-Whitney: MannwhitneyuResult(statistic=np.float64(48257337.5), pvalue=np.float64(0.0))


**Chosen test:** one-sided Welch two-sample t-test (Levene: p = 0.02, unequal variances).

- H0: mean (close) ≤ mean (far)
- H1: mean (close) > mean (far)

**Conclusion:** p-value ≈ 0 < 0.05, so we reject H0. Neighborhoods within 0.5 of a school or a hospital have an average median house value of about $247,000, versus about $181,000 for the others. With n ≈ 17,000 the test detects small gaps easily, but this one (about +$66,000) is also large in practical terms. This is an association, not proof of causation: these areas are near Los Angeles and the Bay Area, which are already more expensive.